In [1]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-core
!pip install -q transformers
!pip install -q torch
!pip install -q accelerate
!pip install -q sentencepiece
!pip install -q pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import re
import json
from transformers import pipeline

In [9]:
from transformers import pipeline

classifier = pipeline(
    "text-generation",
    model="distilgpt2",
    max_new_tokens=100
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [10]:
def detect_links(text):
    pattern = r'(https?://\S+|www\.\S+)'
    links = re.findall(pattern, text)

    return {
        "found": len(links) > 0,
        "links": links
    }

In [11]:
def detect_urgency(text):

    urgency_words = [
        "urgent",
        "immediately",
        "act now",
        "limited time",
        "expires",
        "last chance",
        "verify now",
        "update now"
    ]

    found = []

    for word in urgency_words:
        if word.lower() in text.lower():
            found.append(word)

    return found

In [12]:
def detect_scam_keywords(text):

    keywords = [
        "free",
        "winner",
        "lottery",
        "prize",
        "claim reward",
        "gift card",
        "bank account",
        "verify account",
        "click here",
        "password"
    ]

    found = []

    for keyword in keywords:
        if keyword.lower() in text.lower():
            found.append(keyword)

    return found

In [13]:
def llm_analysis(message):

    prompt = f"""
You are a cybersecurity expert.

Analyze the following message and explain:

1. Is it suspicious?
2. Is there phishing?
3. Is there urgency manipulation?
4. Is there a fake reward offer?
5. Give a short explanation.

Message:
{message}
"""

    result = classifier(prompt)

    return result[0]["generated_text"]

In [14]:
def threat_analysis_agent(message):

    links = detect_links(message)

    urgency = detect_urgency(message)

    scam_keywords = detect_scam_keywords(message)

    llm_result = llm_analysis(message)

    risk_score = 0

    if links["found"]:
        risk_score += 30

    risk_score += len(urgency) * 10

    risk_score += len(scam_keywords) * 10

    risk_score = min(risk_score, 100)

    if risk_score > 70:
        risk_level = "HIGH"

    elif risk_score > 40:
        risk_level = "MEDIUM"

    else:
        risk_level = "LOW"

    return {
        "risk_level": risk_level,
        "risk_score": risk_score,
        "links_detected": links["links"],
        "urgency_words": urgency,
        "suspicious_keywords": scam_keywords,
        "llm_explanation": llm_result
    }

In [15]:
sample_message = """
Congratulations!

You have won a $500 Amazon Gift Card.

Claim your reward immediately.

Click here:
https://fake-reward.com

Offer expires today.
"""

result = threat_analysis_agent(sample_message)

print(json.dumps(result, indent=4))

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{
    "risk_level": "MEDIUM",
    "risk_score": 70,
    "links_detected": [
        "https://fake-reward.com"
    ],
    "urgency_words": [
        "immediately",
        "expires"
    ],
    "suspicious_keywords": [
        "gift card",
        "click here"
    ],
    "llm_explanation": "\nYou are a cybersecurity expert.\n\nAnalyze the following message and explain:\n\n1. Is it suspicious?\n2. Is there phishing?\n3. Is there urgency manipulation?\n4. Is there a fake reward offer?\n5. Give a short explanation.\n\nMessage:\n\nCongratulations!\n\nYou have won a $500 Amazon Gift Card.\n\nClaim your reward immediately.\n\nClick here:\nhttps://fake-reward.com\n\nOffer expires today.\n\n3. Will the reward be as high as $1,000?\n4. Does the reward go to you?\n5. Does the reward go to you?\n6. Is there a threat?\n7. Is there a chance of a scam?\n8. Is there a chance of a scam?\n9. Is there a chance of a scam?\n10. Does the reward go to you?\n11. Is there a chance of a scam?\n12. Will"
}
